# M3.6 · 强化学习与生成式

> 强化学习把交易建模为"状态—动作—奖励"的序贯决策；生成式方向（如 AlphaGen）用 RL 自动生成因子表达式。
> 本节自实现 tabular Q-learning 跑通交易决策，并讲清生成式因子的原理。

## 一、MDP 与 Q-learning

**马尔可夫决策过程（MDP）：**
- 状态 S：当前可观测信息（如收益分档）；
- 动作 A：空仓/持有；
- 奖励 R：动作带来的下一期收益；
- 目标：最大化累计折扣奖励。

**Q-learning 更新：**
$$Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\big]$$

**防前视要点：** t 时刻的状态只含 t 及之前信息，奖励取 t+1 期收益，两者错开一期。

> 生成式因子（AlphaGen）：把因子表达式当作动作、IC/收益当作奖励，用 RL 在算子空间搜索表达式，
> 本质是"用强化学习做因子挖掘"，与我们 numpy 版遗传规划异曲同工。


## 二、加载数据并离散化状态

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
import qlib
from qlib.data import D
from l2.config import load_universe
import config
from model_lab2 import rl
qlib.init(provider_uri=str(config.QLIB_DAILY_DIR), joblib_backend='threading')
codes = load_universe('hs300')
raw = D.features(codes, ['$close'], start_time='2018-01-01', freq='day')
cl = raw['$close'].unstack('instrument')
one = cl.iloc[:,0].dropna()
ret = one.pct_change(fill_method=None).dropna().values
states = rl.discretize(ret[:-1], 3)   # 状态基于 t 期
print('状态档:', np.unique(states), '样本数:', len(states))

## 三、训练 Q-learning

In [ ]:
agent = rl.QLearningAgent(n_actions=2, lr=0.1, gamma=0.95, eps=0.3)
strat_r, bh_r, actions = rl.train_qlearning(agent, states, ret[1:])
print('策略累计收益=%.4f 买入持有=%.4f' % (strat_r, bh_r))
print('持有动作占比=%.3f 学到的Q表状态数=%d' % (actions.mean(), len(agent.q)))
# 查看各状态的 Q 值
for s in sorted(agent.q):
    print('状态%d Q(空仓)=%.3f Q(持有)=%.3f' % (s, agent.q[s][0], agent.q[s][1]))

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
# 重建净值
fwd = ret[1:][:len(actions)]
nav_s = np.cumprod(1 + actions*fwd)
nav_b = np.cumprod(1 + fwd)
fig, ax = plt.subplots(figsize=(10,4.6))
ax.plot(nav_s, label='RL 策略', color='#4f46e5')
ax.plot(nav_b, label='买入持有', color='#9ca3af')
ax.set_title('RL 策略 vs 买入持有净值'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,2.8))
ax.fill_between(range(len(actions)), actions, color='#f87171', alpha=0.6)
ax.set_yticks([0,1]); ax.set_ylabel('动作'); ax.set_title('各时点持仓动作（1=持有）')
plt.tight_layout(); plt.show()

## 四、严谨性说明（重要）

- 上述收益是**样本内**结果：Q 表在同一段历史上一边探索一边学习，再在该段评估，存在乐观偏差；
- 真实评估应做 **walk-forward**：只用过去训练 Q 表，在后续未见区间执行并滚动；
- 只有 3 个离散状态，信息有限，这里重在理解 RL 机制，而非追求可上线策略。

## 五、小结与面试要点

**面试高频：**
- 讲一下 Q-learning 的更新公式？（答：时序差分，朝 r+γmaxQ(s') 目标更新）
- 探索与利用怎么平衡？（答：epsilon-greedy，训练早期多探索、后期多利用）
- RL 用在交易里怎么防前视？（答：状态只含历史、奖励取下一期，严格错开）
- AlphaGen 是什么？（答：用RL把因子表达式当动作、IC当奖励，搜索生成因子）
- 为什么 RL 在实盘量化里不普遍？（答：奖励稀疏、非平稳、易过拟合、样本内偏差大）

## 动手练习
1. 离散化状态并训练 Q-learning；
2. 对比策略与买入持有净值；
3. 思考如何改造成 walk-forward 外推。

---
下一节 M3.7 时间序列交叉验证与超参优化。